# BÀI TẬP LỚN: HỆ THỐNG DỰ BÁO TRỄ CHUYẾN BAY & SĂN VÉ MÁY BAY GIÁ RẺ NỘI ĐỊA VIỆT NAM
### Môn học: Nhập môn Khoa học Dữ liệu (IT4930 / IT4142) - SoICT, Đại học Bách Khoa Hà Nội (HUST)
---
## MỤC LỤC BÁO CÁO THỰC NGHIỆM
1. **Giai đoạn A: Thu thập & Tích hợp Dữ liệu Đa nguồn**
2. **Giai đoạn B: Làm sạch Dữ liệu & Thống kê Mô tả Đơn biến**
3. **Giai đoạn C: Kỹ nghệ Đặc trưng & Trực quan hóa Đa biến**
4. **Giai đoạn D: Huấn luyện Mô hình, Regularization & Khảo sát Overfitting**
5. **Giai đoạn E: Tổng kết Hiệu năng & Triển khai Ứng dụng Web Dashboard**

## 1. Giai đoạn A: Thu thập & Tích hợp Dữ liệu Đa nguồn
- Xây dựng các **Wrappers** chuẩn hóa nguồn dữ liệu chuyến bay nội địa và trạm khí tượng sân bay (Nội Bài HAN, Tân Sơn Nhất SGN, Đà Nẵng DAD).
- Thiết kế **Lược đồ trung gian (Mediated Schema)** và thực hiện phép kết nối hợp nhất theo khóa đa thuộc tính `(timestamp, origin)`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_collection.fetch_flights import generate_vietnam_flights_stream
from src.data_collection.fetch_weather_api import generate_airport_weather_stream, fetch_airport_weather_live
from src.data_integration.integrate import DataIntegrationEngine

# 1. Khởi tạo và tích hợp dữ liệu đa nguồn qua Mediated Schema
engine = DataIntegrationEngine()
df_integrated = engine.integrate()
print("Cấu trúc Lược đồ trung gian (Mediated Schema):", list(df_integrated.columns))
df_integrated.head()

## 2. Giai đoạn B: Làm sạch Dữ liệu & Thống kê Mô tả Đơn biến
- Kiểm tra và xử lý giá trị khuyết (Missing Values).
- Phát hiện và xử lý ngoại lai (Outliers) theo quy tắc hàng rào Tukey $1.5 \times \text{IQR}$.
- Tính toán các chỉ số thống kê mô tả: Mean, Median, Mode, Variance, Std, Skewness, Kurtosis.

In [ ]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()
print("1. Kiểm tra tỷ lệ khuyết dữ liệu:")
display(cleaner.check_missing_values())

# Tiến hành làm sạch và xử lý ngoại lai theo 1.5xIQR
df_cleaned = cleaner.clean_and_save()

# Thống kê đơn biến cho biến mục tiêu số phút trễ chuyến
stats_delay = cleaner.compute_univariate_statistics('delay_minutes')
pd.DataFrame([stats_delay])

## 3. Giai đoạn C: Kỹ nghệ Đặc trưng & Trực quan hóa Dữ liệu Đa biến
- Trích xuất đặc trưng chu kỳ thời gian: $\sin / \cos$ của giờ bay và ngày trong tuần.
- Đặc trưng trễ chuỗi thời gian: Mức trễ mạng bay giờ trước (`network_delay_lag_1h`) và trung bình trượt 3 giờ.
- Trực quan hóa chuẩn mực: Run Sequence Plot, Lag Plot tự tương quan, Heatmap $24 \times 7$, Ma trận tương quan Pearson, Parallel Coordinates và Biểu đồ đường cong giá vé (Dynamic Pricing).

In [ ]:
from src.preprocessing.feature_engineering import build_features, generate_airfare_pricing_curve_dataset
from src.visualization.eda_plots import generate_all_eda_plots

# Tạo lập tập đặc trưng và dữ liệu giá vé
df_featured = build_features()
df_pricing = generate_airfare_pricing_curve_dataset()

# Xuất bản toàn bộ biểu đồ EDA khoa học vào docs/figures/
generate_all_eda_plots()
print("Đã hoàn thành sinh toàn bộ đặc trưng và biểu đồ EDA!")

## 4. Giai đoạn D: Huấn luyện Mô hình, Regularization & Khảo sát Overfitting
- Phân chia tập dữ liệu theo dòng thời gian liên tục (Time-series Split: 70% Train, 15% Val, 15% Test) chống rò rỉ dữ liệu (Data Leakage).
- Chuẩn hóa `StandardScaler` chỉ khớp (`fit`) trên tập Train.
- So sánh 5 thuật toán: Baseline (Naïve Persistence), Linear Regression, Ridge ($L_2$ Penalty), Random Forest, XGBoost Regressor.
- Thực nghiệm phân tích hiện tượng Overfitting theo độ sâu cây (`max_depth` từ 1 đến 24) làm rõ 3 vùng: Underfitting, Điểm tối ưu, Overfitting.

In [ ]:
from src.models.train_evaluate import train_and_compare_models

metrics_df = train_and_compare_models()
display(metrics_df)

## 5. Kết luận & Triển khai Ứng dụng Web Dashboard
- Mô hình **XGBoost Regressor** đạt độ chính xác tối ưu (MAE nhỏ nhất, khả năng phân loại trễ $\ge 30$ phút đạt độ chính xác cao).
- Hệ thống đã được đóng gói và tích hợp vào Web Dashboard tương tác **Streamlit** (`app/app.py`) phục vụ hành khách tra cứu nguy cơ trễ chuyến và săn vé máy bay giá rẻ.